In [0]:
#pip install numpy

In [0]:
import numpy as np 

def tool_monitor_data_drift(baseline_data: np.ndarray, production_data: np.ndarray, threshold: float = 0.25):
    """Calcula o PSI para detectar Data Drift em produção."""
    def calculate_psi(expected, observed, num_bins=10):
        bins = np.percentile(expected, np.linspace(0, 100, num_bins + 1))
        bins[0] = -np.inf
        bins[-1] = np.inf
        
        freq_esp, _ = np.histogram(expected, bins=bins)
        freq_obs, _ = np.histogram(observed, bins=bins)
        
        pct_esp = np.maximum(freq_esp / len(expected), 1e-4)
        pct_obs = np.maximum(freq_obs / len(observed), 1e-4)
        
        psi = np.sum((pct_obs - pct_esp) * np.log(pct_obs / pct_esp))
        return float(psi)
        
    psi_value = calculate_psi(baseline_data, production_data)
    print(f"[Agente] Índice PSI calculado: {psi_value:.4f}")
    
    if psi_value > threshold:
        print("[Agente] ALERTA: Data Drift severo detectado! Disparando re-treinamento automático.")
        return {"drift_detected": True, "psi": psi_value}
    else:
        print("[Agente] Estabilidade de dados normal (Sem Drift significativo).")
        return {"drift_detected": False, "psi": psi_value}